In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_date, lit

rules = [
    Row(
        rule_id=1,
        rule_name="High Fraud Probability",
        risk_level="HIGH",
        threshold=0.90
    ),
    Row(
        rule_id=2,
        rule_name="Medium Fraud Probability",
        risk_level="MEDIUM",
        threshold=0.50
    ),
    Row(
        rule_id=3,
        rule_name="Low Fraud Probability",
        risk_level="LOW",
        threshold=0.25
    )
]

rules_df = spark.createDataFrame(rules)

rules_df = (
    rules_df
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

display(rules_df)

rule_id,rule_name,risk_level,threshold,effective_from,effective_to,is_current
1,High Fraud Probability,HIGH,0.9,2026-09-26,null,true
2,Medium Fraud Probability,MEDIUM,0.5,2026-09-26,null,true
3,Low Fraud Probability,LOW,0.25,2026-09-26,null,true


In [0]:
(
    rules_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_fraud_rules")
)

print("Initial SCD Type 2 dimension created successfully!")

Initial SCD Type 2 dimension created successfully!


In [0]:
%sql
SELECT *
FROM workspace.default.dim_fraud_rules
ORDER BY rule_id;

rule_id,rule_name,risk_level,threshold,effective_from,effective_to,is_current
1,High Fraud Probability,HIGH,0.9,2026-09-26,null,true
2,Medium Fraud Probability,MEDIUM,0.5,2026-09-26,null,true
3,Low Fraud Probability,LOW,0.25,2026-09-26,null,true


In [0]:
from pyspark.sql.functions import col

In [0]:
from pyspark.sql.functions import current_date, date_sub, lit

rules_df = spark.table("workspace.default.dim_fraud_rules")

# Expire the old HIGH-risk rule
updated_rules = (
    rules_df
    .withColumn(
        "effective_to",
        date_sub(current_date(), 1)
    )
    .withColumn(
        "is_current",
        lit(False)
    )
    .filter(col("rule_id") == 1)
)

# Keep rules 2 and 3 unchanged
unchanged_rules = (
    rules_df
    .filter(col("rule_id") != 1)
)

# Create the new version of rule 1
new_rule = spark.createDataFrame([
    Row(
        rule_id=1,
        rule_name="High Fraud Probability",
        risk_level="HIGH",
        threshold=0.85
    )
])

new_rule = (
    new_rule
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

scd2_df = (
    unchanged_rules
    .unionByName(updated_rules)
    .unionByName(new_rule)
)

display(
    scd2_df.orderBy("rule_id", "effective_from")
)

rule_id,rule_name,risk_level,threshold,effective_from,effective_to,is_current
1,High Fraud Probability,HIGH,0.85,2026-09-26,null,true
1,High Fraud Probability,HIGH,0.9,2026-09-26,2026-09-25,false
2,Medium Fraud Probability,MEDIUM,0.5,2026-09-26,null,true
3,Low Fraud Probability,LOW,0.25,2026-09-26,null,true


In [0]:
(
    scd2_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_fraud_rules")
)

print("SCD Type 2 dimension updated successfully!")

SCD Type 2 dimension updated successfully!


In [0]:
%sql
SELECT
    rule_id,
    rule_name,
    risk_level,
    threshold,
    effective_from,
    effective_to,
    is_current
FROM workspace.default.dim_fraud_rules
ORDER BY rule_id, effective_from;

rule_id,rule_name,risk_level,threshold,effective_from,effective_to,is_current
1,High Fraud Probability,HIGH,0.85,2026-09-26,null,true
1,High Fraud Probability,HIGH,0.9,2026-09-26,2026-09-25,false
2,Medium Fraud Probability,MEDIUM,0.5,2026-09-26,null,true
3,Low Fraud Probability,LOW,0.25,2026-09-26,null,true
